# 04 — Pairs trading (Engle–Granger + z-score)

**Project:** cointegration residual mean-reversion on a synthetic pair.

## Problem
Two related prices may share a common stochastic trend. If the residual is
mean-reverting, can a simple z-score rule harvest the reversion after a lag?

## Method
1. Build a synthetic cointegrated pair with `make_cointegrated_pair`.
2. Test with `engle_granger` (OLS hedge ratio + ADF on residuals).
3. Trade with `PairsStrategy` (entry/exit on rolling z-score).

## Modules
- `quant_lab.pairs`
- `quant_lab.data.sample`


In [ ]:
from quant_lab.data import make_cointegrated_pair
from quant_lab.pairs import engle_granger, PairsStrategy, PairsSignalConfig
from quant_lab.risk import risk_summary

pair = make_cointegrated_pair(n=600, hedge_ratio=0.9, half_life=15.0, seed=2)
print(pair.head())

coint = engle_granger(pair["y"], pair["x"], use_log=True)
print(coint.summary())


In [ ]:
cfg = PairsSignalConfig(entry_z=2.0, exit_z=0.5, z_window=60, use_log=True)
strat = PairsStrategy(cfg)

# Fit on first half, trade on second half (simple holdout)
split = len(pair) // 2
train, test = pair.iloc[:split], pair.iloc[split:]
strat.fit(train["y"], train["x"])
out = strat.generate(test["y"], test["x"])
print(out.dropna().tail())
print()
metrics = risk_summary(out["pnl"].dropna())
for k, v in metrics.items():
    print(f"{k}: {v}")


In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless-friendly
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
out["spread"].plot(ax=axes[0], title="Residual spread (test)")
out["zscore"].plot(ax=axes[1], title="Rolling z-score")
axes[1].axhline(cfg.entry_z, ls="--", color="gray", lw=1)
axes[1].axhline(-cfg.entry_z, ls="--", color="gray", lw=1)
axes[1].axhline(cfg.exit_z, ls=":", color="gray", lw=1)
axes[1].axhline(-cfg.exit_z, ls=":", color="gray", lw=1)
out["pnl"].cumsum().plot(ax=axes[2], title="Cumulative residual PnL")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Short result
The synthetic pair is constructed to be cointegrated, so ADF typically rejects
the unit-root null. The z-score policy then earns positive residual PnL on the
holdout when half-life and thresholds align — see the cumulative PnL chart.

## Limitations
- Residual PnL is pedagogical (not dollar notionals or borrow costs).
- Hedge ratio fitted once; real pairs need rolling / Kalman updates.
- ADF critical values here are approximate MacKinnon shortcuts.
- Cointegration can break permanently; research/education only.
